# North Carolina PDF parsing walkthrough

NCSLC monthly PDFs use two month-label shapes:

- **Footnoted:** `March1`, `July1` — the trailing digit is a footnote marker, not part of the month.
- **Plain:** `April`, `May`, … — ordinary month names in cumulative/YTD reports.

Later-month PDFs repeat earlier months (year-to-date). The collector keeps the latest file
for each calendar month. This notebook loads saved local fixtures only — no downloads, no SQLite.

In [1]:
from io import BytesIO
from pathlib import Path
import sys

import pandas as pd
import pdfplumber

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.states.north_carolina import (
    build_normalized_row,
    normalize_month_label,
    parse_revenue_pdf,
)

FIXTURES = ROOT / "tests" / "fixtures" / "NC"
PDFS = {
    "March 2024 (footnote)": FIXTURES / "NCSLC-Sports-Betting-Revenue-Report-March-2024.pdf",
    "April 2024 (cumulative)": FIXTURES / "NCSLC-Sports-Betting-Revenue-Report-April-2024.pdf",
    "July 2024 (footnote)": FIXTURES / "NCSLC-Sports-Betting-Revenue-Report-July-2024.pdf",
}
for path in PDFS.values():
    assert path.exists(), path
print("fixtures:", list(PDFS))

fixtures: ['March 2024 (footnote)', 'April 2024 (cumulative)', 'July 2024 (footnote)']


In [2]:
def first_table_label(path: Path) -> str:
    with pdfplumber.open(BytesIO(path.read_bytes())) as pdf:
        text = (pdf.pages[0].extract_text() or "").splitlines()
        tables = pdf.pages[0].extract_tables() or []
    print(f"=== {path.name} ===")
    print("\n".join(text[:7]))
    data_rows = [row for row in (tables[0] if tables else []) if row and row[0] and str(row[0]).strip().lower() not in {"month", "total"}]
    for row in data_rows[:3]:
        label = str(row[0]).strip()
        print(f"label={label!r} -> normalized={normalize_month_label(label)!r}")
    return str(data_rows[0][0]).strip() if data_rows else ""

labels = {name: first_table_label(path) for name, path in PDFS.items()}
labels

=== NCSLC-Sports-Betting-Revenue-Report-March-2024.pdf ===
NORTH CAROLINA | Sports Betting Revenue Report | FY 2024
Promo
Paid Wagering Total Wagering Cancelled/Void Amounts Paid Gross Wagering Estimated
Month Wagering
Revenue Revenue Wagers as Winnings Revenue Tax Proceeds
Revenue
March1 $456,702,632 $202,605,909 $659,308,541 $2,062,025 $590,750,303 $66,496,213 $11,969,318
label='March1' -> normalized='March'
=== NCSLC-Sports-Betting-Revenue-Report-April-2024.pdf ===
NORTH CAROLINA | Sports Betting Revenue Report | FY 2024
Promo
Paid Wagering Total Wagering Cancelled/Void Amounts Paid Gross Wagering Estimated
Month Wagering
Revenue Revenue Wagers as Winnings Revenue Tax Proceeds
Revenue
March1 $456,702,632 $202,605,909 $659,308,541 $2,062,025 $590,750,303 $66,496,213 $11,969,318
label='March1' -> normalized='March'
label='April' -> normalized='April'
=== NCSLC-Sports-Betting-Revenue-Report-July-2024.pdf ===
NORTH CAROLINA | Sports Betting Revenue Report | FY 2025
Promo
Paid Wagering T

{'March 2024 (footnote)': 'March1',
 'April 2024 (cumulative)': 'March1',
 'July 2024 (footnote)': 'July1'}

The April report is cumulative: it lists **March1** (footnote) and **April** (plain label)
on separate rows. Both normalize to month names before money columns are parsed.

In [3]:
retrieved_at = pd.Timestamp.now("UTC")
frames = []
for name, path in PDFS.items():
    for record in parse_revenue_pdf(path.read_bytes()):
        frames.append(
            build_normalized_row(
                record,
                source_url=f"file://{path.name}",
                source_file=str(path.relative_to(ROOT)).replace("\\", "/"),
                source_sha256="notebook-demo",
                retrieved_at=retrieved_at.to_pydatetime(),
            )
        )

tidy = pd.concat(frames, ignore_index=True)
tidy[["period_start", "handle", "gross_revenue", "tax", "reported_revenue_name"]]

,period_start,handle,gross_revenue,tax,reported_revenue_name
0,2024-03-01,659308541.0,66496213.0,11969318.0,Gross Wagering Revenue
1,2024-03-01,659308541.0,66496213.0,11969318.0,Gross Wagering Revenue
2,2024-04-01,648934226.0,105251672.0,18945301.0,Gross Wagering Revenue
3,2024-07-01,340375354.0,42226041.0,7600687.0,Gross Wagering Revenue


In [4]:
april_path = PDFS["April 2024 (cumulative)"]
parsed = {(r["year"], r["month"]): r for r in parse_revenue_pdf(april_path.read_bytes())}
assert len(parsed) == 2
march = parsed[(2024, 3)]
april = parsed[(2024, 4)]
print("March handle/GGR/tax:", march["handle"], march["gross_revenue"], march["tax"])
print("April handle/GGR/tax:", april["handle"], april["gross_revenue"], april["tax"])
assert march["handle"] == 659_308_541 and april["handle"] == 648_934_226
print("April cumulative report reconciled; Total row excluded")

March handle/GGR/tax: 659308541.0 66496213.0 11969318.0
April handle/GGR/tax: 648934226.0 105251672.0 18945301.0
April cumulative report reconciled; Total row excluded


## Promoted into `north_carolina.py`

- `ROW_RE` uses named groups: `month`, optional `footnote`, and `values`.
- Money is parsed from the `values` group — never from the footnote digit.
- `normalize_month_label()` accepts `March`, `March1`, and rejects `Total`.
- Column positions unchanged: handle = Total Wagering Revenue, gross = Gross Wagering Revenue, tax = Estimated Tax Proceeds.
- FY headings still infer calendar year (`FY 2024` → March/April 2024; `FY 2025` → July 2024).